## Imports

In [2]:
import pandas as pd
import re
from datetime import datetime, timedelta

## Initial Formatting

In [4]:
# Load raw data
df = pd.read_csv("data/mcp_desc_all_jan_22.csv")

# --- 1. Remove emojis / non-ascii ---
def remove_non_ascii(text):
    if isinstance(text, str):
        return re.sub(r'[^\x00-\x7F]+', '', text)
    return text

for col in ["title", "description", "use_cases", "key_features"]:
    if col in df.columns:
        df[col] = df[col].apply(remove_non_ascii).str.strip()


# --- 2. Drop junk / empty text ---
# heuristics for "useless" descriptions
def is_useless_desc(text):
    if not isinstance(text, str):
        return True
    txt = text.lower().strip()
    if len(txt.split()) < 5:
        return True
    bad_patterns = ["mirror", "demo", "test"]
    return any(p in txt for p in bad_patterns)

df["desc_is_useless"] = df["description"].apply(is_useless_desc)
df["has_use_cases"] = df["use_cases"].notna() & (df["use_cases"].str.strip() != "")

# keep if we have use cases OR meaningful description
df = df[(~df["desc_is_useless"]) | (df["has_use_cases"])].reset_index(drop=True)

# --- 3. Clean whitespace, collapse doubles ---
df = df.replace({r'\s+': ' '}, regex=True)

# --- 4. Standardize uploaded column (MM-DD-YYYY format) ---
def parse_relative_date(text):
    if not isinstance(text, str):
        return None

    m = re.search(r"(\d+)\s*(year|month|week|day)s?\s*ago", text.lower())
    if not m:
        return None

    num, unit = int(m.group(1)), m.group(2)
    now = datetime.now()

    if unit == "year":
        dt = now - timedelta(days=num * 365)
    elif unit == "month":
        dt = now - timedelta(days=num * 30)
    elif unit == "week":
        dt = now - timedelta(weeks=num)
    elif unit == "day":
        dt = now - timedelta(days=num)
    else:
        return None

    return dt.strftime("%m-%d-%Y")  # formatted as MM-DD-YYYY

df["uploaded_clean"] = df["uploaded"].apply(parse_relative_date)


# --- 5. Combine description + use cases for LLM input ---
def combine_text(desc, use):
    parts = []
    if isinstance(desc, str) and desc.strip():
        parts.append(f"Description: {desc.strip()}")
    if isinstance(use, str) and use.strip():
        parts.append(f"Use cases: {use.strip()}")
    return "; ".join(parts)

df["text_for_llm"] = df.apply(lambda x: combine_text(x["description"], x["use_cases"]), axis=1)

def combine_text_with_features(desc, use, features):
    parts = []
    if isinstance(desc, str) and desc.strip():
        parts.append(f"Description: {desc.strip()}")
    if isinstance(features, str) and features.strip():
        parts.append(f"Key features: {features.strip()}")
    if isinstance(use, str) and use.strip():
        parts.append(f"Use cases: {use.strip()}")
    return "; ".join(parts)

df["text_for_llm_2"] = df.apply(
    lambda x: combine_text_with_features(
        x["description"], x["use_cases"], x["key_features"]
    ),
    axis=1
)

# --- 6. Drop rows with no usable text at all ---
df = df[df["text_for_llm"].str.strip() != ""]

# --- Deduplicate on final LLM input ---
before = len(df)
df = df.drop_duplicates(subset=["text_for_llm"]).reset_index(drop=True)
after = len(df)

print(f"🧹 Removed {before - after} duplicate rows")

# --- 7. Optional: flag short entries ---
df["len_text"] = df["text_for_llm"].str.len()
df = df[df["len_text"] > 40]  # drop really short junk

# drop columns we don't need anymore
cols_to_drop = [
    "uploaded",          # raw text version
    "use_cases",         # now merged into text_for_llm
    "description",       # same, merged
    "desc_is_useless",   # helper boolean
    "has_use_cases"      # helper boolean
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# --- 8. Save cleaned dataset ---
df.to_csv("data/mcp_desc_all_jan_22_cleaned.csv", index=False)
print(f"✅ Cleaned dataset saved ({len(df)} rows)")


🧹 Removed 11 duplicate rows
✅ Cleaned dataset saved (8953 rows)


### Average Char Count

In [5]:
import pandas as pd

df = pd.read_csv("data/mcp_desc_all_jan_22_cleaned.csv")

for col in ["text_for_llm", "text_for_llm_2"]:
    if col in df.columns:
        avg_len = df[col].astype(str).str.len().mean()
        print(f"{col}: average character count = {avg_len:.1f}")
    else:
        print(f"{col}: column not found")


print("\nMedian lengths:")
for col in ["text_for_llm", "text_for_llm_2"]:
    if col in df.columns:
        med_len = df[col].astype(str).str.len().median()
        print(f"{col}: median character count = {med_len:.1f}")


text_for_llm: average character count = 293.3
text_for_llm_2: average character count = 518.0

Median lengths:
text_for_llm: median character count = 268.0
text_for_llm_2: median character count = 490.0
